In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, ArrayType, StructType, StructField, LongType, DoubleType

spark = SparkSession.builder \
    .appName("MyDigitalTwin-BehavioralClustering") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# ── Config centrale ────────────────────────────────────────────────────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.path.dirname('__file__'), '../../..')))
from config import WAREHOUSE

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

def read_table(name):
    return spark.read.parquet(os.path.join(WAREHOUSE, name))

Warehouse: /opt/spark/warehouse


26/04/05 13:09:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## PARTIE B — Behavioral Clustering

On rassemble toutes les activités avec leurs features temporelles : heure, jour de la semaine, plateforme, poids d'interaction.

**Features retenues (V1 finale)** : `hour_sin/cos`, `weekday_sin/cos`, `weight`, `platform_ohe`

> **Note V2 testée** : retrait de la plateforme pour des profils purement temporels → 6 clusters quasi-identiques (doublons "Soir Semaine" × 2, "Après-midi Weekend" × 2, Silhouette 0.33). La plateforme est un signal comportemental réel — mode Spotify journée ≠ soirée multi-plateforme. V1 conservée.

In [2]:
# ── B1. CHARGEMENT DES FEATURES COMPORTEMENTALES ──────────────────────────────
# Features retenues : hour (cyclique), weekday (cyclique), platform (OHE), weight
# Sources avec colonnes heure + jour disponibles

youtube = read_table("youtube_watch") \
    .select(F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"),
            F.lit("youtube").alias("platform"),
            F.col("interaction_weight").cast(FloatType()).alias("weight"))

google = read_table("google_searches") \
    .select(F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"),
            F.lit("google").alias("platform"),
            F.lit(1.0).cast(FloatType()).alias("weight"))

chrome = read_table("google_chrome") \
    .select(F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"),
            F.lit("chrome").alias("platform"),
            F.lit(1.0).cast(FloatType()).alias("weight"))

spotify = read_table("spotify_streams") \
    .select(F.col("listen_hour").alias("hour"),
            F.col("listen_weekday").alias("weekday"),
            F.lit("spotify").alias("platform"),
            F.col("interaction_weight").cast(FloatType()).alias("weight"))

# Netflix : pas de colonne hour → 21h par défaut (visionnage soir)
netflix = read_table("netflix_views") \
    .select(F.lit(21).cast(IntegerType()).alias("hour"),
            F.col("watch_weekday").alias("weekday"),
            F.lit("netflix").alias("platform"),
            F.col("interaction_weight").cast(FloatType()).alias("weight"))

# TikTok : 234k rows → limit 2000 (éviter de noyer les autres sources)
tiktok = read_table("tiktok_watch") \
    .select(F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"),
            F.lit("tiktok").alias("platform"),
            F.col("interaction_weight").cast(FloatType()).alias("weight")) \
    .limit(2000)

# Instagram : 17k rows → limit 2000
ig_likes = read_table("instagram_likes") \
    .select(F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"),
            F.lit("instagram").alias("platform"),
            F.col("interaction_weight").cast(FloatType()).alias("weight")) \
    .limit(2000)

behavioral_raw = youtube.union(google).union(chrome).union(spotify) \
    .union(netflix).union(tiktok).union(ig_likes) \
    .filter(F.col("hour").isNotNull() & F.col("weekday").isNotNull())

print(f"Total events comportementaux : {behavioral_raw.count():,}")
behavioral_raw.groupBy("platform").count().orderBy(F.desc("count")).show()

Total events comportementaux : 112,273


+---------+-----+
| platform|count|
+---------+-----+
|   google|55854|
|  spotify|33972|
|  youtube|13821|
|  netflix| 4288|
|   tiktok| 2000|
|instagram| 2000|
|   chrome|  338|
+---------+-----+



In [3]:
# ── B2. FEATURE ENGINEERING COMPORTEMENTAL ────────────────────────────────────
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

# Encoder la plateforme (catégorielle → OHE)
indexer = StringIndexer(inputCol="platform", outputCol="platform_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCol="platform_idx", outputCol="platform_ohe", dropLast=False)

# Features cycliques temporelles
behavioral_feat = behavioral_raw \
    .withColumn("hour_sin",    F.sin(2 * 3.14159 * F.col("hour") / 24)) \
    .withColumn("hour_cos",    F.cos(2 * 3.14159 * F.col("hour") / 24)) \
    .withColumn("weekday_sin", F.sin(2 * 3.14159 * F.col("weekday") / 7)) \
    .withColumn("weekday_cos", F.cos(2 * 3.14159 * F.col("weekday") / 7)) \
    .withColumn("weight_norm", F.coalesce(F.col("weight"), F.lit(1.0)).cast("double"))

indexer_model = indexer.fit(behavioral_feat)
behavioral_feat = indexer_model.transform(behavioral_feat)

encoder_model = encoder.fit(behavioral_feat)
behavioral_feat = encoder_model.transform(behavioral_feat)

# VectorAssembler : features temporelles + plateforme
assembler_beh = VectorAssembler(
    inputCols=["hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "weight_norm", "platform_ohe"],
    outputCol="beh_raw_features"
)
behavioral_feat = assembler_beh.transform(behavioral_feat)

scaler_beh = StandardScaler(inputCol="beh_raw_features", outputCol="beh_features",
                             withMean=False, withStd=True)
scaler_model = scaler_beh.fit(behavioral_feat)
behavioral_feat = scaler_model.transform(behavioral_feat)

print("Features comportementales construites.")
behavioral_feat.select("hour", "weekday", "platform", "beh_features").show(5, truncate=80)

Features comportementales construites.
+----+-------+--------+--------------------------------------------------------------------------------+
|hour|weekday|platform|                                                                    beh_features|
+----+-------+--------+--------------------------------------------------------------------------------+
|  19|      7| youtube|(13,[0,1,2,3,4,7],[-1.5908258895735936,0.36222616395323237,-7.539013746468581...|
|  19|      7| youtube|(13,[0,1,2,3,4,7],[-1.5908258895735936,0.36222616395323237,-7.539013746468581...|
|  19|      7| youtube|(13,[0,1,2,3,4,7],[-1.5908258895735936,0.36222616395323237,-7.539013746468581...|
|  19|      7| youtube|(13,[0,1,2,3,4,7],[-1.5908258895735936,0.36222616395323237,-7.539013746468581...|
|  19|      7| youtube|(13,[0,1,2,3,4,7],[-1.5908258895735936,0.36222616395323237,-7.539013746468581...|
+----+-------+--------+--------------------------------------------------------------------------------+
only showing top

In [4]:
# ── B3. KMEANS COMPORTEMENTAL (k=6) ───────────────────────────────────────────
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

K_BEHAVIORAL = 6

kmeans_beh = KMeans(
    featuresCol="beh_features",
    predictionCol="beh_cluster",
    k=K_BEHAVIORAL,
    seed=42,
    maxIter=50
)

print(f"Training K-Means behavioral (k={K_BEHAVIORAL})...")
km_beh_model = kmeans_beh.fit(behavioral_feat)
beh_df = km_beh_model.transform(behavioral_feat)

evaluator_beh = ClusteringEvaluator(featuresCol="beh_features", predictionCol="beh_cluster")
sil_beh = evaluator_beh.evaluate(beh_df)
print(f"Silhouette Score (behavioral): {sil_beh:.4f}")

beh_df.groupBy("beh_cluster").count().orderBy("beh_cluster").show()

Training K-Means behavioral (k=6)...


26/04/05 13:09:32 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Silhouette Score (behavioral): 0.3181
+-----------+-----+
|beh_cluster|count|
+-----------+-----+
|          0|13821|
|          1|31774|
|          2| 2000|
|          3|33972|
|          4|  338|
|          5|30368|
+-----------+-----+



In [5]:
# ── B4. CARACTERISATION DES CLUSTERS COMPORTEMENTAUX ─────────────────────────
# Optimisation : une seule agrégation Spark au lieu d'une boucle Python
# (avant : 24 jobs Spark = 4 actions × 6 clusters ; après : 2 jobs)

from pyspark.sql.window import Window

# ① Statistiques globales par cluster (1 job)
beh_agg = beh_df.groupBy("beh_cluster").agg(
    F.count("*").alias("item_count"),
    F.round(F.avg("hour"), 1).alias("avg_hour"),
    F.round(F.avg("weekday"), 1).alias("avg_weekday"),
).orderBy("beh_cluster")

beh_agg.show()

# ② Top 3 plateformes par cluster via window function (1 job)
platform_window = Window.partitionBy("beh_cluster").orderBy(F.desc("cnt"))
platform_top3 = (
    beh_df.groupBy("beh_cluster", "platform").agg(F.count("*").alias("cnt"))
    .withColumn("rn", F.row_number().over(platform_window))
    .filter(F.col("rn") <= 3)
    .groupBy("beh_cluster")
    .agg(F.collect_list("platform").alias("top_platforms"))
)

platform_by_cluster = {row["beh_cluster"]: row["top_platforms"] for row in platform_top3.collect()}

# ③ Construire beh_cluster_info depuis les résultats collectés
beh_cluster_info = []
for row in beh_agg.collect():
    cluster_id = row["beh_cluster"]
    avg_h  = row["avg_hour"]  or 0.0
    avg_wd = row["avg_weekday"] or 0.0
    count  = row["item_count"]
    top_plt = platform_by_cluster.get(cluster_id, [])

    h = round(avg_h)
    if 5 <= h < 12:    period = "Matin"
    elif 12 <= h < 18: period = "Après-midi"
    elif 18 <= h < 23: period = "Soir"
    else:              period = "Nuit"

    day_type = "Weekend" if round(avg_wd) >= 5 else "Semaine"

    beh_cluster_info.append({
        "cluster_id":   cluster_id,
        "item_count":   count,
        "avg_hour":     float(avg_h),
        "avg_weekday":  float(avg_wd),
        "time_period":  period,
        "day_type":     day_type,
        "top_platforms": top_plt,
    })

    print(f"[Beh Cluster {cluster_id}] {count:,} items | {period} · {day_type} | Plateformes: {top_plt}")


+-----------+--------+-----------+----------+
|beh_cluster|avg_hour|avg_weekday|item_count|
+-----------+--------+-----------+----------+
|          0|    15.7|        3.9|     13821|
|          1|    18.6|        4.0|     31774|
|          2|    13.7|        3.5|      2000|
|          3|    12.7|        4.0|     33972|
|          4|    19.5|        3.3|       338|
|          5|    12.9|        3.9|     30368|
+-----------+--------+-----------+----------+



[Beh Cluster 0] 13,821 items | Après-midi · Semaine | Plateformes: ['youtube']


[Beh Cluster 1] 31,774 items | Soir · Semaine | Plateformes: ['google', 'netflix', 'instagram']


[Beh Cluster 2] 2,000 items | Après-midi · Semaine | Plateformes: ['tiktok']


[Beh Cluster 3] 33,972 items | Après-midi · Semaine | Plateformes: ['spotify']


[Beh Cluster 4] 338 items | Soir · Semaine | Plateformes: ['chrome']


[Beh Cluster 5] 30,368 items | Après-midi · Semaine | Plateformes: ['google']


In [6]:
# ── B5. LABELLING DES CLUSTERS COMPORTEMENTAUX ────────────────────────────────
# Labels basés sur les résultats V1 (avec plateforme dans les features)
# Cluster → plateforme dominante + moment de la journée

BEH_LABELS = {
    0: {"label": "📺 YouTube · Après-midi",    "emoji": "📺"},   # 13k items, youtube, après-midi semaine
    1: {"label": "🛋️ Soirée connectée",         "emoji": "🛋️"},  # 31k items, google+netflix+instagram, soir semaine
    2: {"label": "📱 TikTok · Scroll",          "emoji": "📱"},   # 2k items, tiktok (limité), après-midi semaine
    3: {"label": "🎵 Spotify · Journée",        "emoji": "🎵"},   # 33k items, spotify, après-midi semaine
    4: {"label": "💻 Navigation · Soir",        "emoji": "💻"},   # 338 items, chrome, soir semaine
    5: {"label": "🔍 Recherches · Journée",     "emoji": "🔍"},   # 30k items, google, après-midi semaine
}

print("Labels définis.")

Labels définis.


In [7]:
# ── B6. ECRITURE behavioral_clusters ──────────────────────────────────────────
from pyspark.sql.types import DoubleType

beh_rows = []
for info in beh_cluster_info:
    cid = info["cluster_id"]
    beh_rows.append((
        cid,
        BEH_LABELS.get(cid, {}).get("label", f"Profil {cid}"),
        BEH_LABELS.get(cid, {}).get("emoji", "❓"),
        float(info["avg_hour"]),
        float(info["avg_weekday"]),
        info["time_period"],
        info["day_type"],
        info["top_platforms"],
        info["item_count"]
    ))

schema_beh = StructType([
    StructField("cluster_id",    IntegerType(), False),
    StructField("label",         StringType(),  False),
    StructField("emoji",         StringType(),  True),
    StructField("avg_hour",      DoubleType(),  True),
    StructField("avg_weekday",   DoubleType(),  True),
    StructField("time_period",   StringType(),  True),
    StructField("day_type",      StringType(),  True),
    StructField("top_platforms", ArrayType(StringType()), True),
    StructField("item_count",    LongType(),    True),
])

beh_clusters_df = spark.createDataFrame(beh_rows, schema_beh)

out_path_beh = os.path.join(WAREHOUSE, "behavioral_clusters")
beh_clusters_df.write.mode("overwrite").parquet(out_path_beh)

print(f"Ecrit dans : {out_path_beh}")
beh_clusters_df.show(truncate=50)

Ecrit dans : /opt/spark/warehouse/behavioral_clusters
+----------+-----------------------+-----+--------+-----------+-----------+--------+----------------------------+----------+
|cluster_id|                  label|emoji|avg_hour|avg_weekday|time_period|day_type|               top_platforms|item_count|
+----------+-----------------------+-----+--------+-----------+-----------+--------+----------------------------+----------+
|         0|📺 YouTube · Après-midi|   📺|    15.7|        3.9| Après-midi| Semaine|                   [youtube]|     13821|
|         1|   🛋️ Soirée connectée|  🛋️|    18.6|        4.0|       Soir| Semaine|[google, netflix, instagram]|     31774|
|         2|     📱 TikTok · Scroll|   📱|    13.7|        3.5| Après-midi| Semaine|                    [tiktok]|      2000|
|         3|   🎵 Spotify · Journée|   🎵|    12.7|        4.0| Après-midi| Semaine|                   [spotify]|     33972|
|         4|   💻 Navigation · Soir|   💻|    19.5|        3.3|       Soir| Semai

In [8]:
spark.stop()
print("Spark session fermée. Notebook terminé.")

Spark session fermée. Notebook terminé.
